# V28 — 3D rocket landing

**How to use:** Runtime > Change runtime type > **T4 GPU** > Save. Then Runtime > **Run all**. That is the entire workflow.

**Auto-resume:** every run of Cell 2 reads `MyDrive/rocket_v28/models/master_v28_latest.zip` from Drive. If it is there, training continues from that step. If not, training starts from step 0. The resume logic is `src/train_v28_mujoco.py` lines 352-376 (`resume = LATEST.exists() and not FRESH`, `SAC.load(LATEST)`, `load_replay_buffer`).

**Different Google account:** Drive folders are per-account. If you open this on a new account and want to continue a previous run, share `MyDrive/rocket_v28/` from the original account to the new one in Drive (right-click > Share). Then Run all works unchanged.

## Cell 1: setup — GPU, Drive, dependencies, code

In [ ]:
# --- GPU check -----------------------------------------------------------
import subprocess, torch, os
print(subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout.strip() or 'no GPU')

# --- Drive ---------------------------------------------------------------
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/rocket_v28'
os.makedirs(f'{DRIVE}/models', exist_ok=True)
os.makedirs(f'{DRIVE}/logs', exist_ok=True)
print('output folder:', DRIVE)

# --- Dependencies --------------------------------------------------------
# tensorflow is broken on Colab Python 3.13 and SB3 does not need it; removing
# it avoids a slow import stall on every launch of the training script.
%pip install -q mujoco "stable-baselines3>=2.4" "gymnasium>=1.0" tensorboard imageio imageio-ffmpeg
%pip uninstall -y -q tensorflow tensorflow-cpu 2>/dev/null
import mujoco, gymnasium, stable_baselines3
print(f'mujoco {mujoco.__version__} | gymnasium {gymnasium.__version__} | sb3 {stable_baselines3.__version__} | cuda {torch.cuda.is_available()}')

# --- Code (always pull latest) -------------------------------------------
REPO = 'https://github.com/rajivagarwal221/Rocket-Land.git'
if not os.path.isdir('/content/rocket'):
    !git clone --depth 1 {REPO} /content/rocket
else:
    !cd /content/rocket && git fetch --depth 1 origin main -q && git reset --hard origin/main -q
%cd /content/rocket
!git log --oneline -1

## Cell 2: train — resumes automatically

Before launching, this cell reads the Drive checkpoint and prints exactly where training will start.
The script loads it with `SAC.load("master_v28_latest.zip")` and also loads `master_v28_buffer.pkl` if present.
Re-run the cell any time — after a disconnect, next day, different runtime — it picks up where it stopped.

In [ ]:
import os, json
from pathlib import Path

TOTAL_STEPS = 6_000_000
LATEST          = Path(DRIVE) / 'models' / 'master_v28_latest.zip'
LATEST_MANIFEST = Path(DRIVE) / 'models' / 'master_v28_latest_manifest.json'
BUFFER          = Path(DRIVE) / 'models' / 'master_v28_buffer.pkl'
BEST            = Path(DRIVE) / 'models' / 'master_v28_best.zip'

print('--- Drive checkpoint preflight -----------------------------------------')
print(f'checkpoint : {LATEST} {"EXISTS" if LATEST.exists() else "missing (will start fresh)"}')
print(f'buffer     : {BUFFER}  {"EXISTS" if BUFFER.exists() else "missing (SB3 will refill)"}')
print(f'best       : {BEST}    {"EXISTS" if BEST.exists() else "missing (first eval creates it)"}')
if LATEST_MANIFEST.exists():
    m = json.loads(LATEST_MANIFEST.read_text())
    print(f'last saved at step {m["total_timesteps"]:,} / {TOTAL_STEPS:,} '
          f'({m["total_timesteps"]/TOTAL_STEPS:.1%}) -- will resume here')
elif LATEST.exists():
    print('checkpoint found but no manifest; SAC.load will still resume from the .zip')
else:
    print(f'no checkpoint on Drive -- training starts from step 0 of {TOTAL_STEPS:,}')
print('------------------------------------------------------------------------')

os.environ.update(
    V28_OUT   = DRIVE,
    V28_STEPS = str(TOTAL_STEPS),
    V28_NENVS = '4',
    V28_SEED  = '2828',
    V28_FRESH = '0',   # set to '1' only if you want to wipe Drive and start over
)
!python src/train_v28_mujoco.py

## Cell 3: evaluate the best checkpoint

Scores `master_v28_best.zip` over 500 seeded episodes at the ship bar (full difficulty, 0.5 m pad). The CSV is written to Drive next to the checkpoint.

In [ ]:
import os
BEST = f'{DRIVE}/models/master_v28_best.zip'
if not os.path.isfile(BEST):
    print(f'no best checkpoint yet at {BEST}. Run Cell 2 first until at least one evaluation prints "<- new best".')
else:
    !python src/evaluate_3d.py --model {BEST} --episodes 500 --device cuda         --out {DRIVE}/logs/v28_best_audit.csv